# DebateFloor — GRPO Training Notebook
### Meta PyTorch × Scaler Hackathon Grand Finale | April 25–26, 2026

This notebook runs **`train/train_minimal.py`** — the same script that produced the reward curve in the README.

| | |
|---|---|
| **Model** | `Qwen/Qwen2.5-0.5B-Instruct` (default) — or override in Cell 2 |
| **Runtime** | T4 GPU (free Colab tier), ~15 min |
| **Method** | TRL `GRPOTrainer` — no Unsloth required |
| **Reward** | `training_reward()` — simple scalar, stable GRPO gradients |
| **Output** | `docs/reward_curve.svg`, `docs/component_shift.svg`, `reports/training_summary.json` |

Based on [CoCA arXiv:2603.05881](https://arxiv.org/abs/2603.05881)

---
▶ **Run all cells top to bottom. Restart runtime after Cell 1, then continue from Cell 2.**

## Cell 1 — Install & clone
Run once. **Restart runtime after this cell.**

In [ ]:
import os

# Clone repo if not already present
if not os.path.exists('debateFloor'):
    !git clone https://github.com/AniketAslaliya/debateFloor.git

# Install dependencies (same as train/requirements.txt)
!pip install -q trl>=0.9.0 transformers>=4.40.0 peft accelerate datasets wandb matplotlib requests

print('\n✅ Done. NOW RESTART THE RUNTIME (Runtime → Restart session), then run Cell 2 onward.')

## Cell 2 — Configure run
**Edit only this cell** to customise your training run. Everything else is automatic.

In [ ]:
import os, sys

# ── Move into repo ──────────────────────────────────────────────
REPO_DIR = '/content/debateFloor'   # standard Colab path after clone
if not os.path.exists(REPO_DIR):
    REPO_DIR = 'debateFloor'        # fallback if already in /content
os.chdir(REPO_DIR)
sys.path.insert(0, '.')
print(f'Working directory: {os.getcwd()}')

# ── ✏️  Edit these values ────────────────────────────────────────

# Model — any HF causal LM that fits in ~14GB VRAM on T4
# Options (free T4):  Qwen/Qwen2.5-0.5B-Instruct  (~1GB, 15 min)
#                     Qwen/Qwen2.5-1.5B-Instruct  (~3GB, 30 min)
#                     Qwen/Qwen2.5-3B-Instruct    (~6GB, 45 min)
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

EPISODES   = 100    # training episodes (100 = 15 min on T4, 200 = 30 min)
EPOCHS     = 2      # 2–3 recommended
BATCH_SIZE = 2      # keep at 2 for T4

WANDB_API_KEY = ''  # paste your WandB key here, or leave empty to skip
WANDB_ENTITY  = 'aniketaslaliya-lnmiit'  # your WandB username / org

# HF token — only needed if you want to push the trained model to the Hub
HF_TOKEN = ''

# ── Inject config into environment (train_minimal.py reads these) ─
os.environ['WANDB_API_KEY']  = WANDB_API_KEY
os.environ['WANDB_ENTITY']   = WANDB_ENTITY
os.environ['HF_TOKEN']       = HF_TOKEN

print(f'Model:    {MODEL_NAME}')
print(f'Episodes: {EPISODES} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}')
print(f'WandB:    {"enabled → " + WANDB_ENTITY if WANDB_API_KEY else "disabled (no key)"}')

## Cell 3 — Sanity-check imports
Confirms the repo modules and GPU are visible before spending 15 minutes training.

In [ ]:
import torch
from server.calibration_grader import training_reward, CALIBRATION_MATRIX
from server.claim_generator import generate_episode_pool

# GPU check
assert torch.cuda.is_available(), 'No GPU found — switch runtime to T4 GPU.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Reward sanity check
r = training_reward('deny_claim', 'MED', 'deny_claim', 2, 1, True)
print(f'Calibration matrix: {CALIBRATION_MATRIX}')
print(f'Reward (deny MED correct, 2 flags): {r:.2f}  ← expect ~1.25')

# Episode generator check
eps = generate_episode_pool(count=3)
print(f'Episode pool check: generated {len(eps)} episodes OK')
print(f'  Sample: {eps[0].claim_id} | {eps[0].fraud_type} | {eps[0].difficulty}')

print('\n✅ All checks passed — ready to train.')

## Cell 4 — Run training

Calls `train_minimal.py` with the config from Cell 2. Outputs:
- `docs/reward_curve.svg` — loss + reward curve
- `docs/component_shift.svg` — before/after component scores
- `reports/training_summary.json` — full log history
- `debatefloor_checkpoint/` — model checkpoint

Expected reward: starts at ~−0.34, rises to ~+0.83 by step 100.

In [ ]:
import importlib, train.train_minimal as tm

# Patch config from Cell 2 into the module
tm.MODEL_NAME  = MODEL_NAME
tm.EPISODES    = EPISODES
tm.EPOCHS      = EPOCHS
tm.BATCH_SIZE  = BATCH_SIZE
tm.USE_WANDB   = bool(WANDB_API_KEY)
tm.WANDB_KEY   = WANDB_API_KEY
tm.WANDB_ENTITY = WANDB_ENTITY

# Also update dtype flags (may change if model is different)
tm.HAS_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
tm.USE_FP16 = torch.cuda.is_available() and not tm.HAS_BF16
tm.DTYPE    = torch.bfloat16 if tm.HAS_BF16 else torch.float16

print(f'dtype: {tm.DTYPE} | fp16={tm.USE_FP16} | bf16={tm.HAS_BF16}')
print(f'Starting training: {EPISODES} episodes × {EPOCHS} epochs...\n')

tm.main()

## Cell 5 — View results inline

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

# Show reward curve
if Path('docs/reward_curve.svg').exists():
    display(Image('docs/reward_curve.svg'))

# Show component shift
if Path('docs/component_shift.svg').exists():
    display(Image('docs/component_shift.svg'))

# Print key numbers
summary = json.loads(Path('reports/training_summary.json').read_text())
rewards = [r['reward'] for r in summary['log_history'] if 'reward' in r and 'step' in r]
if rewards:
    print(f'\nReward — start: {rewards[0]:.3f} → end: {rewards[-1]:.3f} → peak: {max(rewards):.3f}')

cs = summary.get('component_shift', {})
before = cs.get('before', {})
after  = cs.get('after', {})
if before and after:
    print('\nComponent shift (before → after):')
    for k in before:
        b, a = before[k], after[k]
        arrow = '↑' if a > b else '↓' if a < b else '→'
        print(f'  {k:<28} {b:+.3f}  →  {a:+.3f}  {arrow}')

## Cell 6 — (Optional) Push checkpoint to HuggingFace Hub

In [ ]:
# Only runs if HF_TOKEN was set in Cell 2
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

    from transformers import AutoModelForCausalLM, AutoTokenizer
    ckpt = './debatefloor_checkpoint'
    model = AutoModelForCausalLM.from_pretrained(ckpt)
    tok   = AutoTokenizer.from_pretrained(ckpt)
    hub_name = f'AniketAsla/debatefloor-grpo-{MODEL_NAME.split("/")[-1].lower()}'
    model.push_to_hub(hub_name)
    tok.push_to_hub(hub_name)
    print(f'Pushed to https://huggingface.co/{hub_name}')
else:
    print('HF_TOKEN not set — skipping Hub push.')
    print('Set HF_TOKEN in Cell 2 and re-run this cell to push.')

## Cell 7 — (Optional) Get WandB run URL to paste into README

In [ ]:
if WANDB_API_KEY:
    import wandb
    api = wandb.Api()
    runs = api.runs(f'{WANDB_ENTITY}/debatefloor-insurance-rl', order='-created_at')
    latest = next(iter(runs), None)
    if latest:
        url = f'https://wandb.ai/{WANDB_ENTITY}/debatefloor-insurance-rl/runs/{latest.id}'
        print(f'\n✅ Your specific WandB run URL (paste this into README):')
        print(f'   {url}')
    else:
        print('No runs found in project yet.')
else:
    print('WandB not enabled. Set WANDB_API_KEY in Cell 2 to get a public run URL.')